In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool
from langchain.chat_models import init_chat_model
from langchain.messages import SystemMessage,HumanMessage,AIMessage
from DashScope import messages
load_dotenv()   #.env

# 通过api加载llm
# llm = init_chat_model(
#     model="qwen3.5-flash",
#     model_provider="openai",
#     api_key=os.getenv("DASHSCOPE_API_KEY"),
#     base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
# )

# 通过ollama加载llm
llm = init_chat_model(
    model="qwen3.5:9b",
    model_provider="ollama",
)

response = llm.invoke("你是谁？")

import os
import numpy as np
import onnxruntime
import torch
from PIL import Image
from torchvision import transforms
from transformers import AutoTokenizer

# 基础配置
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
PROVIDERS = [
    ('CUDAExecutionProvider', {'device_id': 0, 'arena_extend_strategy': 'kSameAsRequested'}),
    'CPUExecutionProvider',
]

# 图像预处理
IMG_TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 全局变量，用于缓存模型
_IMAGE_ORT_SESSION = None
_TEXT_ORT_SESSION = None
_TOKENIZER = None

def init_retrieval_models(image_model_path='image_model_flickr.onnx',
                          text_model_path='text_model_flickr.onnx',
                          tokenizer_path="BAAI/bge-small-en-v1.5"):
    """初始化并缓存模型"""
    global _IMAGE_ORT_SESSION, _TEXT_ORT_SESSION, _TOKENIZER
    _IMAGE_ORT_SESSION = onnxruntime.InferenceSession(image_model_path, providers=PROVIDERS)
    _TEXT_ORT_SESSION = onnxruntime.InferenceSession(text_model_path, providers=PROVIDERS)
    _TOKENIZER = AutoTokenizer.from_pretrained(tokenizer_path, local_files_only=True)
    print("Retrieval models loaded successfully.")

init_retrieval_models()

def hamming_distance_np(a, b):
    return 0.5 * (b.shape[1] - np.dot(a, b.T))

@tool
def text_to_image_retrieval(query_text: str, image_folder: str = r"D:\Learning\datasets\mirflickr-qt\test_T2I", top_k: int = 5) -> list:
    """
    根据文本描述在指定文件夹中搜索匹配的图片。

    Args:
        query_text: 用户输入的描述文本（必须先翻译为英文再输入！！）
        image_folder: 存放待检索图片的文件夹路径，不需要填入，已经有默认值了！
        top_k: 返回结果的数量
    """
    if _TEXT_ORT_SESSION is None:
        raise RuntimeError("Models not initialized. Call init_retrieval_models() first.")

    # 1. 文本特征提取 (Encoding)
    text_token = _TOKENIZER([query_text], padding=True, truncation=True, return_tensors="np")
    text_ort_inputs = {
        'input_ids': text_token['input_ids'].astype(np.int64),
        'attention_mask': text_token['attention_mask'].astype(np.int64),
    }
    text_hash = _TEXT_ORT_SESSION.run(['text_output'], text_ort_inputs)[0]
    text_hash = np.sign(text_hash)

    # 2. 遍历文件夹提取图片特征
    img_paths = [os.path.join(image_folder, f) for f in os.listdir(image_folder)
                 if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

    img_hashes = []
    valid_paths = []

    for path in img_paths:
        try:
            img = Image.open(path).convert('RGB')
            img_tensor = IMG_TRANSFORM(img).unsqueeze(0).numpy() # 转为numpy给ONNX

            image_ort_inputs = {'image_input': img_tensor}
            h = _IMAGE_ORT_SESSION.run(['image_output'], image_ort_inputs)[0]
            img_hashes.append(np.sign(h))
            valid_paths.append(path)
        except Exception as e:
            continue # 忽略损坏的图片

    if not img_hashes:
        return []

    # 3. 计算汉明距离并排序
    all_hashes = np.concatenate(img_hashes, axis=0)
    dists = hamming_distance_np(text_hash, all_hashes).flatten()
    top_indices = np.argsort(dists)[:top_k]

    return [valid_paths[idx] for idx in top_indices]

@tool
def image_to_text_retrieval(query_image_path: str, text_database_path: str = r"D:\Learning\datasets\mirflickr-qt\test_I2T.txt", top_k: int = 5) -> list:
    """
    根据上传的图片在文本库中搜索最符合的描述。

    Args:
        query_image_path: 查询图片的路径
        text_database_path: 存放文本库的 .txt 文件路径，不需要填入，已经有默认值了！
        top_k: 返回结果的数量
    """

    if _IMAGE_ORT_SESSION is None:
        raise RuntimeError("Models not initialized. Call init_retrieval_models() first.")

    # 1. 图片特征提取
    img = Image.open(query_image_path).convert('RGB')
    img_tensor = IMG_TRANSFORM(img).unsqueeze(0).numpy()
    img_hash = _IMAGE_ORT_SESSION.run(['image_output'], {'image_input': img_tensor})[0]
    img_hash = np.sign(img_hash)

    # 2. 读取并处理文本库
    with open(text_database_path, 'r', encoding='utf-8') as f:
        texts = [line.strip() for line in f if line.strip()]

    text_hashes = []
    valid_texts = []

    for txt in texts:
        try:
            text_token = _TOKENIZER([txt], padding=True, truncation=True, return_tensors="np")
            text_ort_inputs = {
                'input_ids': text_token['input_ids'].astype(np.int64),
                'attention_mask': text_token['attention_mask'].astype(np.int64),
            }
            h = _TEXT_ORT_SESSION.run(['text_output'], text_ort_inputs)[0]
            text_hashes.append(np.sign(h))
            valid_texts.append(txt)
        except Exception:
            continue

    if not text_hashes:
        return []

    # 3. 计算距离并排序
    all_text_hashes = np.concatenate(text_hashes, axis=0)
    dists = hamming_distance_np(img_hash.squeeze(0), all_text_hashes).flatten()
    top_indices = np.argsort(dists)[:top_k]

    return [valid_texts[idx] for idx in top_indices]

# result = text_to_image_retrieval.run("bugs")
# print(result)

# result = image_to_text_retrieval.run(r"D:\Learning\datasets\mirflickr-qt\test_T2I\im198.jpg")
# print(result)

{"status_code": 200, "request_id": "1169d955-43fd-9532-8e02-f25c1f494fba", "code": "", "message": "", "output": {"text": null, "finish_reason": null, "choices": [{"finish_reason": "stop", "message": {"role": "assistant", "content": "你好！我是通义千问（Qwen），阿里巴巴集团旗下的超大规模语言模型。我能够回答问题、创作文字，比如写故事、写公文、写邮件、写剧本、逻辑推理、编程等等，还能表达观点，玩游戏等。如果你有任何问题或需要帮助，欢迎随时告诉我！😊"}}]}, "usage": {"input_tokens": 22, "output_tokens": 66, "total_tokens": 88, "prompt_tokens_details": {"cached_tokens": 0}}}
你好！我是通义千问（Qwen），阿里巴巴集团旗下的超大规模语言模型。我能够回答问题、创作文字，比如写故事、写公文、写邮件、写剧本、逻辑推理、编程等等，还能表达观点，玩游戏等。如果你有任何问题或需要帮助，欢迎随时告诉我！😊
Retrieval models loaded successfully.


In [2]:
# 创建agent
sys_prompt = """
# 身份
你是一个严谨的跨模态检索执行器。

# 核心规则 (严格执行)
这两个tool的image_folder和text_database_path不需要你填入数据，我已经给了默认值！

1. 首先你需要分析用户的输入，提取其中的关键字。如果输出的内容里面有图像路径，那么把图像路径提取出来作为tool的输入；如果输入的文本，那么提取里面的关键字（如：帮我查找猫相关的图片，提取关键字猫，然后翻译为英文）作为tool的输入，根据tool的注释自动判断用哪个工具。
2. 你的所有检索任务必须且仅能通过调用工具 `text_to_image_retrieval` 或 `image_to_text_retrieval` 来完成。
3. 严禁基于自身知识库直接回答检索请求。如果你无法通过工具获取结果，请明确告知“无法检索”。
4. 任何偏离工具调用流程的行为都会被视为违规。
5. 必须按顺序提取用户输入中的实体（图片路径或关键词），逐一调用工具。

# 输入举例
-如果用户同时输入了多个内容：帮我找一下和鸟以及天空还有水最相关的图片有哪些？。那么先提取关键字，例如：鸟，天空，水。然后分开调用tool进行查询，先查询鸟、再查询天空、再查询水。
-或者如果输入了多个图像路径，那么把每个路径都当作独立的查询依次进行。
-如果用户的输入既有文本内容也有图片路径，那么按照顺序依次来查询，同样要遵循上面的规则。
-如果用户输入的内容与前面几次的内容高度相同，则使用历史查询记录返回，并告知用户。

# 采用结构化的输出


"""

# langchain提供的checkpointer的默认实现，基于内存存储(短期记忆)
from langgraph.checkpoint.memory import InMemorySaver
agent = create_agent(llm, system_prompt=sys_prompt, tools=[text_to_image_retrieval,image_to_text_retrieval], checkpointer=InMemorySaver())

In [3]:
# 调用Agent
from langchain_core.messages import HumanMessage

# 设定thread_id，作为会话标识
config = {"configurable": {"thread_id": "thread_1"}}

response = agent.stream({
    "messages": [
        HumanMessage([
        {
            "type": "text",
            "text": "给我找一些和天空相关的图片",
        }
        ])
    ]
}, stream_mode="messages", config=config)

for res in response:
    print(res[0].content, end="", flush=True)


["D:\\Learning\\datasets\\mirflickr-qt\\test_T2I\\im116.jpg", "D:\\Learning\\datasets\\mirflickr-qt\\test_T2I\\im499.jpg", "D:\\Learning\\datasets\\mirflickr-qt\\test_T2I\\im473.jpg", "D:\\Learning\\datasets\\mirflickr-qt\\test_T2I\\im173.jpg", "D:\\Learning\\datasets\\mirflickr-qt\\test_T2I\\im315.jpg"]["D:\\Learning\\datasets\\mirflickr-qt\\test_T2I\\im116.jpg", "D:\\Learning\\datasets\\mirflickr-qt\\test_T2I\\im499.jpg", "D:\\Learning\\datasets\\mirflickr-qt\\test_T2I\\im473.jpg", "D:\\Learning\\datasets\\mirflickr-qt\\test_T2I\\im173.jpg", "D:\\Learning\\datasets\\mirflickr-qt\\test_T2I\\im315.jpg"]已找到与“天空”相关的图片，以下是检索结果：

1. `D:\\Learning\\datasets\\mirflickr-qt\\test_T2I\\im116.jpg`
2. `D:\\Learning\\datasets\\mirflickr-qt\\test_T2I\\im499.jpg`
3. `D:\\Learning\\datasets\\mirflickr-qt\\test_T2I\\im473.jpg`
4. `D:\\Learning\\datasets\\mirflickr-qt\\test_T2I\\im173.jpg`
5. `D:\\Learning\\datasets\\mirflickr-qt\\test_T2I\\im315.jpg`

需要更多结果或其他关键词的查询，请随时告知！